In [0]:
from pyspark.sql.functions import col, lit, when, current_timestamp
from datetime import datetime

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA silver")

In [0]:
df_bronze = spark.table("scottish_housing_ws.bronze.scottish_gdp")

df_bronze.select("industry_sector").distinct().show(30, truncate=False)

df_bronze.select("Measurement").distinct().show(10, truncate=False)

In [0]:
from pyspark.sql.types import DateType
import re

def parse_quarter_date(date_code):
    # Expects format like '2004-Q4'
    match = re.match(r"(\d{4})-Q(\d)", date_code)
    if not match:
        return None
    year = int(match.group(1))
    quarter = int(match.group(2))
    month = {1: 1, 2: 4, 3: 7, 4: 10}[quarter]
    return datetime(year, month, 1).date()

parse_quarter_udf = udf(parse_quarter_date, DateType())

In [0]:
df_silver = (
    df_bronze
    .filter(col("industry_sector") == "Total Gross Value Added (GVA) (Section A-T)")
    .withColumn("report_date", parse_quarter_udf(col("DateCode")))
    .withColumnRenamed("DateCode", "year_quarter")
    .withColumnRenamed("Measurement", "measure_type")
    .withColumnRenamed("Units", "units")
    .withColumnRenamed("Value", "value")
    .drop("FeatureCode", "FeatureName", "FeatureType", "industry_sector")
    .select("report_date", "year_quarter", "measure_type", "units", "value")
)

df_silver.printSchema()

In [0]:
df_silver.orderBy("report_date", "measure_type").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import min, max, count
df_silver.agg(
    count("*").alias("row_count"),
    min("report_date").alias("earliest"),
    max("report_date").alias("latest")
).show()

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.silver.scottish_gdp")
)

In [0]:
# Filtered to Total GDP sector only. All four measure types retained:
# index, q-on-q, q-on-q-year-ago, 4q-on-4q.
# report_date is the first day of each quarter.
# year_quarter retains the original '2004-Q4' label for readability.
# Fully qualified table name used on write due to UDF catalog reset gotcha.